# AIQuant — HFT Statistical Arbitrage Framework
**AegisFintech | Apache 2.0 | [github.com/AegisFintech/AIQuant](https://github.com/AegisFintech/AIQuant)**

This notebook runs the full AIQuant pipeline on Google Colab with **zero local setup**:
1. Clone the repo & install dependencies
2. Fetch real BTCUSD 1-minute data from Binance (free, no API key)
3. Build 100+ features (technical, microstructure, statistical arbitrage)
4. Run the Backtrader backtest with Kalman StatArb strategy
5. Run the paper trading engine (live Binance data)
6. Display performance charts inline

---
**Colab Compatibility Notes:**
- ✅ Free tier (CPU): Full backtest + paper trading — works perfectly
- ✅ T4 GPU: ML model training (XGBoost/LightGBM) is faster
- ⚠️  RAM: Free tier has 12GB RAM. For 90-day backtests use `--days 30`. For full 90 days, use **Colab Pro (25GB RAM)**
- ⚠️  Session timeout: Colab disconnects after ~90min idle. Use paper trading `--bars N` to set a finite run


## Step 1 — Clone Repository & Install Dependencies

In [ ]:
# Clone the AIQuant repository
!git clone https://github.com/AegisFintech/AIQuant.git
%cd AIQuant
!pip install -q -r requirements.txt
print('\n✅ Installation complete')

In [ ]:
# Verify key packages
import sys
sys.path.insert(0, '/content/AIQuant')

import numpy as np
import pandas as pd
import backtrader as bt
import numba
print(f'NumPy  {np.__version__}')
print(f'Pandas {pd.__version__}')
print(f'Backtrader {bt.__version__}')
print(f'Numba  {numba.__version__}')
print('\n✅ All packages ready')

## Step 2 — Configuration
Edit the cell below to customise your run.

In [ ]:
# ─── USER CONFIGURATION ───────────────────────────────────────────────────
PAIR         = 'BTCUSDT'   # Trading pair (BTCUSDT, ETHUSDT, SOLUSDT, etc.)
DAYS         = 30          # Lookback window in days
              #   Free Colab (12GB RAM):  use 30 days
              #   Colab Pro  (25GB RAM):  use 90 days
              #   High-RAM server:        use 180+ days
CAPITAL      = 100_000     # Starting capital in USD
KELLY_FRAC   = 0.5         # Kelly fraction (0.5 = Half-Kelly, recommended)
PAPER_BARS   = 20          # Number of bars for paper trading demo (0 = live loop)
# ──────────────────────────────────────────────────────────────────────────

print(f'Config: {PAIR} | {DAYS} days | ${CAPITAL:,} capital | Kelly={KELLY_FRAC}')

## Step 3 — Fetch Real BTCUSD 1-Minute Data

In [ ]:
import os, sys, time, requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta, timezone

DATA_DIR = Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch_binance_1m(pair='BTCUSDT', days=30):
    """Fetch 1m OHLCV from Binance public API. No API key required."""
    base_url = 'https://api.binance.com/api/v3/klines'
    end_ms   = int(datetime.now(timezone.utc).timestamp() * 1000)
    start_ms = end_ms - days * 86_400_000
    all_bars = []
    limit    = 1000
    since_ms = start_ms

    print(f'Fetching {pair} 1m data ({days} days = {days*1440:,} bars)...')
    while since_ms < end_ms:
        resp = requests.get(base_url, params={
            'symbol': pair, 'interval': '1m',
            'startTime': since_ms, 'limit': limit
        }, timeout=30)
        data = resp.json()
        if not data or isinstance(data, dict):
            break
        all_bars.extend(data)
        since_ms = data[-1][0] + 60_000
        print(f'  {len(all_bars):,} bars fetched...', end='\r')
        time.sleep(0.1)

    arr = np.array(all_bars, dtype=object)
    df  = pd.DataFrame({
        'open':   arr[:, 1].astype(np.float64),
        'high':   arr[:, 2].astype(np.float64),
        'low':    arr[:, 3].astype(np.float64),
        'close':  arr[:, 4].astype(np.float64),
        'volume': arr[:, 5].astype(np.float64),
    }, index=pd.to_datetime(arr[:, 0].astype(np.int64), unit='ms', utc=True))
    df.index.name = 'timestamp'
    df = df[~df.index.duplicated(keep='first')].sort_index()
    cache = DATA_DIR / f'{pair}_1m.parquet'
    df.to_parquet(cache, compression='snappy')
    print(f'\n✅ {len(df):,} bars saved to {cache}')
    return df

df_raw = fetch_binance_1m(PAIR, DAYS)
print(df_raw.tail())

## Step 4 — Feature Engineering (100+ Features)

In [ ]:
from aiquant.utils.fast_math import warmup
print('Warming up Numba JIT compiler...')
warmup()
print('✅ Numba ready')

from aiquant.features import build_full_feature_set

print('Building features...')
t0 = time.time()
df_feat = build_full_feature_set(df_raw)
elapsed = time.time() - t0

print(f'✅ {len(df_feat.columns)} features built on {len(df_feat):,} bars in {elapsed:.1f}s')
print(f'Memory usage: {df_feat.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df_feat.iloc[-3:, :10]

## Step 5 — Signal Generation

In [ ]:
from aiquant.strategies.ensemble import StrategyEnsemble

ensemble = StrategyEnsemble()
signals  = ensemble.generate_signals(df_feat)

n_long  = int((signals['final_signal'] == 1).sum())
n_short = int((signals['final_signal'] == -1).sum())
n_flat  = int((signals['final_signal'] == 0).sum())

print(f'✅ Signals generated: {len(signals):,} bars')
print(f'   LONG : {n_long:,} ({n_long/len(signals)*100:.1f}%)')
print(f'   SHORT: {n_short:,} ({n_short/len(signals)*100:.1f}%)')
print(f'   FLAT : {n_flat:,} ({n_flat/len(signals)*100:.1f}%)')

## Step 6 — Backtest (Backtrader)

In [ ]:
import backtrader as bt
import matplotlib
matplotlib.use('Agg')  # Colab-safe backend
import matplotlib.pyplot as plt
from IPython.display import display, Image

Path('results').mkdir(exist_ok=True)

# ── Backtrader Strategy ──────────────────────────────────────────────────
class AIQuantStrategy(bt.Strategy):
    params = dict(capital=CAPITAL)

    def __init__(self):
        self._trade_count = 0
        self._wins = 0
        self._pnl_list = []

    def next(self):
        bar_idx = len(self.data) - 1
        if bar_idx >= len(signals):
            return
        sig = int(signals['final_signal'].iloc[bar_idx])
        pos = self.position.size

        if sig == 1 and pos <= 0:
            self.close()
            size = (self.broker.getcash() * 0.95) / self.data.close[0]
            self.buy(size=size)
        elif sig == -1 and pos >= 0:
            self.close()
            size = (self.broker.getcash() * 0.95) / self.data.close[0]
            self.sell(size=size)
        elif sig == 0 and pos != 0:
            self.close()

    def notify_trade(self, trade):
        if trade.isclosed:
            self._trade_count += 1
            self._pnl_list.append(trade.pnlcomm)
            if trade.pnlcomm > 0:
                self._wins += 1

# ── Run Cerebro ──────────────────────────────────────────────────────────
df_bt = df_feat[['open','high','low','close','volume']].dropna().copy()
df_bt.index = df_bt.index.tz_localize(None)

data_feed = bt.feeds.PandasData(dataname=df_bt)
cerebro   = bt.Cerebro(stdstats=False)
cerebro.adddata(data_feed)
cerebro.addstrategy(AIQuantStrategy)
cerebro.broker.setcash(CAPITAL)
cerebro.broker.setcommission(commission=0.00035)

print(f'Running backtest on {len(df_bt):,} bars...')
t0 = time.time()
results = cerebro.run()
elapsed = time.time() - t0
strat   = results[0]

final_val   = cerebro.broker.getvalue()
total_ret   = (final_val - CAPITAL) / CAPITAL * 100
n_trades    = strat._trade_count
win_rate    = strat._wins / n_trades * 100 if n_trades > 0 else 0
pnl_arr     = np.array(strat._pnl_list)
avg_win     = float(pnl_arr[pnl_arr > 0].mean()) if (pnl_arr > 0).any() else 0
avg_loss    = float(pnl_arr[pnl_arr < 0].mean()) if (pnl_arr < 0).any() else 0

print(f'\n{'─'*50}')
print(f'BACKTEST RESULTS  ·  {PAIR}')
print(f'{'─'*50}')
print(f'Initial Capital   ${CAPITAL:>15,.2f}')
print(f'Final Value       ${final_val:>15,.2f}')
print(f'Total Return      {total_ret:>14.2f}%')
print(f'Total Trades      {n_trades:>15,}')
print(f'Win Rate          {win_rate:>14.1f}%')
print(f'Avg Win           ${avg_win:>+14.2f}')
print(f'Avg Loss          ${avg_loss:>+14.2f}')
print(f'Runtime           {elapsed:>14.1f}s')
print(f'{'─'*50}')

## Step 7 — Performance Chart

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
for ax in axes.flat:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#c9d1d9')
    ax.spines[:].set_color('#30363d')

# 1. Price + signals
ax1 = axes[0, 0]
close_arr = df_feat['close'].values
ax1.plot(close_arr, color='#58a6ff', linewidth=0.6, label='BTC Close')
sig_arr = signals['final_signal'].values
long_idx  = np.where(sig_arr == 1)[0]
short_idx = np.where(sig_arr == -1)[0]
ax1.scatter(long_idx,  close_arr[long_idx],  marker='^', color='#3fb950', s=8, alpha=0.6, label='Long')
ax1.scatter(short_idx, close_arr[short_idx], marker='v', color='#f85149', s=8, alpha=0.6, label='Short')
ax1.set_title(f'{PAIR} Price + Signals', color='#c9d1d9', fontweight='bold')
ax1.legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=8)
ax1.grid(True, alpha=0.2, color='#30363d')

# 2. PnL distribution
ax2 = axes[0, 1]
if len(pnl_arr) > 0:
    wins   = pnl_arr[pnl_arr > 0]
    losses = pnl_arr[pnl_arr < 0]
    ax2.hist(wins,   bins=30, color='#3fb950', alpha=0.7, label=f'Wins ({len(wins)})')
    ax2.hist(losses, bins=30, color='#f85149', alpha=0.7, label=f'Losses ({len(losses)})')
    ax2.axvline(0, color='white', linestyle='--', alpha=0.5)
ax2.set_title('PnL Distribution', color='#c9d1d9', fontweight='bold')
ax2.legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=8)
ax2.grid(True, alpha=0.2, color='#30363d')

# 3. Hurst Exponent (regime)
ax3 = axes[1, 0]
if 'hurst' in df_feat.columns:
    hurst_arr = df_feat['hurst'].values
    ax3.plot(hurst_arr, color='#d2a8ff', linewidth=0.6)
    ax3.axhline(0.45, color='#3fb950', linestyle='--', alpha=0.7, label='Mean-rev threshold (0.45)')
    ax3.axhline(0.55, color='#f85149', linestyle='--', alpha=0.7, label='Trending threshold (0.55)')
    ax3.axhline(0.5,  color='white',   linestyle=':',  alpha=0.4, label='Random walk (0.5)')
    ax3.set_ylim(0, 1)
ax3.set_title('Hurst Exponent (Regime)', color='#c9d1d9', fontweight='bold')
ax3.legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=7)
ax3.grid(True, alpha=0.2, color='#30363d')

# 4. Kalman Z-Score
ax4 = axes[1, 1]
if 'kalman_zscore' in df_feat.columns:
    kz = df_feat['kalman_zscore'].values
    ax4.plot(kz, color='#ffa657', linewidth=0.5)
    ax4.axhline( 1.8, color='#3fb950', linestyle='--', alpha=0.7, label='Long entry (+1.8)')
    ax4.axhline(-1.8, color='#f85149', linestyle='--', alpha=0.7, label='Short entry (-1.8)')
    ax4.axhline( 0,   color='white',   linestyle=':',  alpha=0.4)
    ax4.set_ylim(-6, 6)
ax4.set_title('Kalman Z-Score (Entry Signal)', color='#c9d1d9', fontweight='bold')
ax4.legend(facecolor='#161b22', labelcolor='#c9d1d9', fontsize=7)
ax4.grid(True, alpha=0.2, color='#30363d')

plt.suptitle(f'AIQuant  ·  {PAIR}  ·  {DAYS}-day Backtest', color='#c9d1d9', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('results/colab_backtest.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('✅ Chart saved to results/colab_backtest.png')

## Step 8 — Paper Trading Demo (Live Binance Data)

In [ ]:
from aiquant.execution.paper_trader import PaperTradingEngine

engine = PaperTradingEngine(
    initial_capital = CAPITAL,
    pair            = PAIR,
    kelly_fraction  = KELLY_FRAC,
    log_dir         = 'logs/paper_trading',
)

print(f'Starting paper trading demo ({PAPER_BARS} bars, ~{PAPER_BARS*2}s)...')
print('─' * 55)

engine.run_replay(
    df          = df_feat,
    signals_df  = signals,
    n_bars      = PAPER_BARS,
    bar_delay   = 2.0,
)

print('─' * 55)
summary = engine.get_summary()
print(f"Final Capital : ${summary['final_capital']:,.2f}")
print(f"Total Trades  : {summary['total_trades']}")
print(f"Realised PnL  : ${summary['total_pnl']:+,.2f}")
print(f"Win Rate      : {summary['win_rate']*100:.1f}%")

## Step 9 — ML Model Training (Optional, GPU-accelerated on T4)

In [ ]:
# Optional: Train the XGBoost + LightGBM ensemble signal model
# Requires at least 30 days of data for meaningful walk-forward CV

from aiquant.models.ml_signal import MLSignalModel

model = MLSignalModel()
print('Training ML ensemble (XGBoost + LightGBM + RF)...')
t0 = time.time()
model.fit(df_feat)
elapsed = time.time() - t0
print(f'✅ Training complete in {elapsed:.1f}s')

ml_signals = model.predict(df_feat)
print(f'ML signal distribution:')
print(ml_signals.value_counts().to_string())

## Step 10 — Save Results & Download
Download the chart and trade log to your local machine.

In [ ]:
from google.colab import files
import os

# Download backtest chart
if os.path.exists('results/colab_backtest.png'):
    files.download('results/colab_backtest.png')

# Download paper trade log
if os.path.exists('logs/paper_trading/paper_trades.csv'):
    files.download('logs/paper_trading/paper_trades.csv')

print('✅ Files downloaded')

---
## Quick Reference

| Command | Description |
|---|---|
| `DAYS = 30` | 30-day window (free Colab safe) |
| `DAYS = 90` | 90-day window (Colab Pro recommended) |
| `PAIR = 'ETHUSDT'` | Switch to Ethereum |
| `KELLY_FRAC = 0.25` | More conservative sizing |
| `PAPER_BARS = 0` | Live paper trading (runs until interrupted) |

**Colab RAM Guide:**
- Free (12GB): `DAYS ≤ 30` (~43,200 bars, ~800MB features)
- Pro (25GB): `DAYS ≤ 90` (~129,600 bars, ~2.4GB features)
- Pro+ (52GB): `DAYS ≤ 180` (~259,200 bars, ~4.8GB features)

**GitHub:** [github.com/AegisFintech/AIQuant](https://github.com/AegisFintech/AIQuant)  
**License:** Apache 2.0 © 2026 AegisFintech
